# Introduction to Quantum Resource Estimation

How many resources does a quantum algorithm actually need to run on a fault-tolerant quantum computer? **Quantum Resource Estimation** (QRE) is what gives us the answer to this question and tells us whether an algorithm is a paper-only result or something we could one day run on real hardware. Good estimates let us drive down the cost of algorithms, compare candidate implementations against each other, and decide which applications are even worth developing.

## The choice of metrics

For a fault-tolerant machine, not all gates cost the same. The operations worth counting are usually the most expensive ones:

- **T gates** &mdash; On a surface-code machine, these require *magic state distillation*, which dominates both the runtime and the footprint. T-count (and/or Toffoli-count) is the headline number in most QRE papers.
- **Toffoli gates** &mdash; Depending on the architecture, these can be native &mdash; implemented directly using CCZ magic states &mdash; or compiled into T gates.
- **Rotations** &mdash; Arbitrary-angle rotation gates, like $R_z(\theta)$, are not native on FTQC architectures and can be only implemented approximately. Each rotation gate needs to be approximated by a whole sequence of T gates in a process called "rotation synthesis", so a rotation gate is really a pile of T gates in disguise.
- **Qubits** &mdash; The number of qubits is considered to be the width of the computation. In practice, we track the **qubit highwater** which is the peak number of qubits live at any one point, including the auxiliary qubits added by the compilation.

> Notice what gates are *missing* from the list: Clifford gates (H, S, CNOT, Paulis). We don't typically count them, as on a fault-tolerant architecture they're comparatively cheap and many can be tracked classically by commuting them through the circuit.
> 
> This rule of thumb may not hold forever. As the field progresses, the gap between non-Clifford and Clifford costs narrows and on some architectures, we may need to start counting Clifford gates too. An example of this progress is "Magic State Cultivation" by Gidney, Shutty & Jones, 2024 ([arXiv:2409.17595](https://arxiv.org/abs/2409.17595)), which describes creating $\ket{T}$ states about as cheaply as a CNOT under certain conditions. However, this topic is out of the scope of this kata.

Additionally, the resource estimator reports several non-standard metrics:

- **Active volume** &mdash; This is a single metric for the logical cost of a program, as described in "Active volume: An architecture for efficient fault-tolerant quantum computers with limited non-local connections" ([arXiv:2211.15465](https://arxiv.org/abs/2211.15465)).
- **Gidney left/right elbows** &mdash; These refer to the two halves of a temporary-AND, introduced in "Halving the cost of quantum addition" ([arXiv:1709.06648](https://arxiv.org/abs/1709.06648)).
- **PPRs and PPMs** &mdash; Pauli Product Rotations and Pauli Product Measurements are multi-qubit rotations and measurements, respectively. They are the operations which lattice surgery actually runs (see "A Game of Surface Codes: Large-Scale Quantum Computing with Lattice Surgery" ([arXiv:1808.02892](https://arxiv.org/abs/1808.02892))).

In this kata, we'll go over the standard gate count metrics and Gidney elbows, leaving the other non-standard metrics to a later, more advanced tutorial.

## The example program we'll use

Throughout this kata, we'll use the same simple program to illustrate the QRE topics we discuss. The code below uses a variety of basic gates: T gates, arbitrary rotations, and a couple of multi-controlled gates. (Note that rotation angles are given in **degrees**.)

In [ ]:
from psiqdk.workbench import QPU, Qubits
from psiqdk.workbench.qre import resource_estimator

def build_circuit(pre_filters=None):
    # We initialize the QPU with more qubits than the 6 the program uses:
    # some gates will need auxiliary qubits for their decompositions.
    qpu = QPU(num_qubits=7, pre_filters=pre_filters)
    reg = Qubits(6, "reg", qpu)

    reg[4].t()
    reg[5].s(reg[4])
    reg[0].rz(123, reg[1:3])
    reg[2].x(reg[3:6])
    reg[1].x(reg[3:6])
    reg[0:3].rz(45)
    return qpu

qpu = build_circuit()
qpu.draw()

## Problem 1. Count rotations and T gates in the "raw" circuit

**Input:** None.

**Goal:**
Count how many rotations (both controlled and uncontrolled) and T gates are in the circuit printed by the code snippet above and fill in your answers below. (Don't consider any gate decompositions yet!)

> It does not make a great deal of sense to count gates as they are shown on the circuit. For one thing, we don't know how we are going to execute multi-controlled gates and whether there are T gates involved. Unfortunately, we can't know that information without at least considering what decomposition we're going to use for them. Even if we knew the decomposition technique, we're not going to count gates manually for any programs of interesting size! So, let's just treat this task as a made-up exercise it is.

In [ ]:
from test_IntroToQRE import problem

@problem
def count_raw_gates():
    # Fill in the values below
    return {"rotations" : ..., "t_gates": ...}

## The `resource_estimator` interface

In practice, rather than counting resources manually, we pass the circuit to the _resource estimator_. `resource_estimator(qpu)` builds an estimator object, and its `resources()` method returns the metrics as a plain dictionary:

In [ ]:
qre = resource_estimator(qpu)
qre.resources()

You can see that the number of T gates (`'t_gates'`) and rotations (`'rotations'`) you counted by hand differ from what you see here. Besides, some other resources on the list probably look unfamiliar.

Both ways to count the resources are correct in some sense. They just answer different questions.

The number you got from looking at the diagram is the count in terms of convenient, high-level gates — the program *as we wrote it*.

The numbers from `resource_estimator` are closer to what we would need to actually run the program on a fault-tolerant quantum computer, after every non-native gate has been broken down into primitives the hardware supports.

The rest of this kata introduces you to the steps involved into getting from the first set of numbers to the second one. But, before we get to it, let's get some practice with using the values returned by the `resources_estimator`.

## Problem 2. Fetch only the resources you need

The `resources()` method returns counts of every resource the estimator tracks, but often you only want to look a handful of those values. In this problem, you'll write a function `relevant_resources` that runs the resource estimator and returns a subset of the resources it counts.

**Input:** The QPU object that stores the circuit you want to analyze.

**Goal:** Get the resource estimates from the resource estimator and return a dictionary with only four of those values: `'t_gates'`, `'toffs'`, `'rotations'`, and `'qubit_highwater'`.

<details>
<summary><strong>Need a hint?</strong></summary>
To fetch a single value out of the dictionary, use <code>qre.resources()["toffs"]</code>.
</details>

In [ ]:
from test_IntroToQRE import problem

@problem
def fetch_relevant_resources(qpu):
    # Write your code here
    return ...

## Filters in Workbench

A **filter** is a compilation pass — a rewrite rule that transforms the circuit into an equivalent one built from lower-level gates. You specify filters when creating a `QPU` instance, and Workbench applies them in order to compile the circuit following the rules specified by them. You can read more about how filters work in the [documentation on Workbench compilation pipeline](https://construct.psiquantum.com/docs/psiqdk-workbench/deepdives/Compilation-Pipeline.html).

Filters usually do one of two jobs:

- **Decomposition** — replace a more complicated gate with its implementation as a sequence of simpler gates (for example, a multi-controlled gate becomes a series of Toffoli and single-controlled gates plus some auxiliary logic, which can then be decomposed further).
- **Optimization** — cancel redundant structure that decomposition exposes (for example, two adjacent rotation gates with the same angle but opposite signs cancel out).

Let's see how they work in our case! As a reminder, here is the example circuit we will be compiling:

In [ ]:
qpu = build_circuit()
qpu.draw()

### Step 1. Decompose multi-controlled gates

The `clean-ladder-filter` turns each multi-controlled gate into a ladder of *Gidney elbows* plus a single-controlled operation.

A **Gidney elbow** is one half of a temporary-AND introduced by Craig Gidney in "Halving the cost of quantum addition" ([arXiv:1709.06648](https://arxiv.org/abs/1709.06648)). A *left elbow* computes the logical AND of two control qubits into a fresh auxiliary qubit; a *right elbow* later uncomputes it with a measurement, far more cheaply than it was computed.

Once we apply this filter and draw the circuit, you'll notice two differences:

1. The elbows which the estimator was already reporting as `'gidney_lelbows'` and `'gidney_relbows'` now appear explicitly.
2. The circuit now uses **extra qubits**, since each left elbow needs a clean auxiliary qubit to store its intermediate AND.

In [ ]:
compilation_filters = [">>clean-ladder-filter>>"]
qpu = build_circuit(pre_filters=compilation_filters)
qpu.draw()

### Step 2. Cancel matching elbows

This is an *optimization* pass &mdash; where a right elbow is immediately followed by a left elbow on the same qubits, the pair cancels. You can see the two elbows between the Toffoli gates cancel each other out.

<center><img src="./images/Elbow-Reduction.svg"></img></center>

In [ ]:
compilation_filters += [">>elbow-reduction-filter>>"]
qpu = build_circuit(pre_filters=compilation_filters)
qpu.draw()

### Step 3. Decompose single-controlled rotations

A controlled rotation is not a native operation for a quantum computer. The standard implementation replaces a controlled $R_z(\theta)$ with two *uncontrolled* half-angle rotations $R_z(\pm\theta/2)$ and two CNOTs which arrange for the two halves to cancel when the control is off and to add up to the full rotation when it's on. So each single-controlled rotation becomes two plain rotations plus two CNOTs.

<center><img src="./images/Single-Control-Decomposition.svg"></img></center>

In [ ]:
compilation_filters += [">>single-control-filter>>"]
qpu = build_circuit(pre_filters=compilation_filters)
qpu.draw()

You can see that the same pass also decomposes the controlled-$S$ gate in our program (`reg[5].s(reg[4])`). The $T$ and $S$ gates are both phase gates:

$$S = \mathrm{phase}(90°), T = \mathrm{phase}(45°)$$

This is where the extra $T$ gates appear in the total count.

> Why *three* of them, though, when the controlled rotation above only took two? Putting a $\mathrm{phase}(45°)$ on both the control and the target gets the $\ket{11}$ phase right ($45° + 45° = 90°$), but it *also* wrongly phases $\ket{01}$ and $\ket{10}$ by $45°$ each. A third $T$-type gate — a $\mathrm{phase}(-45°)$, that is, $T^\dagger$ — is sandwiched between the two CNOTs, so it acts only when exactly one of the two qubits is $\ket{1}$, canceling those spurious phases. This gives us two $T$ and one $T^\dagger$: a controlled-$S$ costs **3 T gates**.

### Step 4. Synthesize the rotations themselves

A generic rotation like $R_z(123°)$ isn't native to a Clifford+T machine. The `rs-synth-filter` approximates each such rotation with a discrete sequence of Clifford+T gates using Ross–Selinger synthesis. Here is what our circuit looks like with this filter applied after all the earlier ones.

In [ ]:
# rs-synth applied on top of the previous filters (still without the toffoli pass).
compilation_filters += [">>rs-synth-filter>>"]
qpu = build_circuit(pre_filters=compilation_filters)
qpu.draw()

In the next problem, you'll take a closer look at the results of using this synthesis filter.

## Problem 3. Synthesize rotations

**Input:** None.

**Goal:**
Fill in the two blanks in the code:

1. Add `>>rs-synth-filter>>` to the QPU's `pre_filters` list.
2. Apply two rotations: $R_z(45°)$ on qubit `r[0]` and $R_z(22.5°)$ on qubit `r[1]`.

Once you complete the task and get the circuit diagram printed, you will see a nice contrast: $R_z(45°)$ comes through as a single $T$ ($45°$ is exactly the $T$ angle), while $R_z(22.5°)$ is a genuine off-grid rotation that unfolds into a long ladder of Clifford+T gates.

In [ ]:
from psiqdk.workbench import QPU, Qubits
from test_IntroToQRE import problem

@problem
def rs_synth_circuit():
    # Add >>rs-synth-filter>> to the pre-filters
    qpu_rot = QPU(num_qubits=2, pre_filters=["..."])
    r = Qubits(2, "reg", qpu_rot)

    with qpu_rot.override_rotation_epsilon(0.15):
        # Apply two rotations: Rz(45°) on r[0] and Rz(22.5°) on r[1]
        ...
    return qpu_rot

> We wrap the rotations in `override_rotation_epsilon(0.15)`.
This method sets the tolerance for rotation synthesis — the additive error we allow when approximating a rotation by a discrete Clifford+T sequence. Choosing a looser tolerance produces a shorter sequence, so we set it to `0.15` purely to make the circuit drawing legible while also fitting on screen.
> 
> This precision isn't free though. The lower the tolerance, the more precise the synthesis, but the more T gates each rotation costs. Try re-running the cell above with smaller values — `1e-2`, then `1e-6` — and watch the $R_z(22.5°)$ decomposition grow. In a real estimate you'd pick the tolerance from the accuracy your algorithm actually needs, not from what makes the circuit fit on screen.

## Problem 4: Count rotations and T gates in the "compiled" circuit

Now, let's return to the question we started with: how to see the circuit on which the resource estimator bases its resource counts?

The resource estimator applies a baseline set of decomposition filters (`clean-ladder-filter` and `single-control-filter`) on its own, which is why the resource counts it returned didn't match the raw diagram. `pre_filters` are applied *on top of* that baseline. There's more on this in the Witness Counter section below.

To see the circuit that matches the resource counts, we just need to pass the right set of filters to our `build_circuit` method. (It passes its `pre_filters=` argument straight to `pre_filters` of the QPU object.)

**Input:** None.

**Goal:**
Count how many rotations, Toffoli gates, T gates, and left Gidney elbows are in the circuit printed by the following code snippet, and fill in your answers below. Remember that $R_z(45°)$ is counted as a T gate!

In [ ]:
qpu = build_circuit(pre_filters=[
    ">>clean-ladder-filter>>",
    ">>single-control-filter>>",
])
qpu.draw()

In [ ]:
from test_IntroToQRE import problem

@problem
def count_compiled_gates():
    # Fill in the values below based on the circuit above
    return {
        "t_gates": ...,           # <-- number of T gates
        "toffs": ...,             # <-- number of Toffoli gates
        "rotations": ...,         # <-- number of rotations
        "gidney_lelbows": ...,    # <-- number of left Gidney elbows
    }

## Under the hood: the Witness Counter

How does `resource_estimator` actually produce those numbers? It *observes* the incoming operations, through a component called the **Witness Counter**.

The Witness Counter is itself a filter (`>>witness>>`), but it is an *analysis* filter &mdash; where the compilation filters rewrite the operation stream, the witness only watches it and is not allowed to change anything. As operations stream past, it builds a **compressed** record of them rather than a flat list. A program with a trillion gates but a lot of repetition still has a small witness record, which is what lets QRE scale to algorithms far too large to write out as explicit circuits.

One subtlety is worth being precise about. To cost an operation, the estimator needs it expressed in terms of fault-tolerant primitives, so before counting, it applies a **default decomposition**: `clean-ladder-filter` and `single-control-filter`. That's why even the very first, "unfiltered" estimate already reported non-zero counts of Gidney elbows and Toffoli gates, and why it didn't match the high-level diagram. Filters you add via `pre_filters` compose *on top of* that baseline; `elbow-reduction`, for instance, isn't in the default set, which is why adding it genuinely drops the elbow count.

Turning the witness record into the resource dictionary is the last step. Each distinct operation is passed through a set of **metric functions** — one per resource (`t_gates`, `toffs`, `rotations`, `measurements`, the elbow counts, `active_volume`, ...) — that say what that operation costs. The costs are summed with the multiplicities from the counter, `qubit_highwater` is read off the peak live-qubit count, and the totals produce exactly the dictionary you saw.

So the complete pipeline is: **operation stream → default decomposition (`clean-ladder`, `single-control`) → witness (compressed operation counts) → per-operation metric functions → resource dictionary.** Filters decide what operations are in the stream, the witness counts them, and the metric functions calculate their cost.

# Architecture-specific considerations

The distribution of resources you get isn't absolute — it depends on which primitives the target hardware treats as fundamental, and that is a per-architecture choice.

Take, for example, Toffoli gates and T gates. Most QRE papers report *one* of these metrics because a Toffoli gate decomposes into a small, fixed number of T gates ($4$ for a Gidney temporary-AND, $7$ for the textbook construction). The two are largely interchangeable, and a single headline number is easier to reason about. That's exactly what the `toffoli-filter` gives us: one more pass on top of the ones from the walkthrough, collapsing the Toffoli/elbow cost into an explicit T-count.

In [ ]:
# We drop >>single-control-filter>> here: it doesn't change the resource counts
# (the estimator applies it internally anyway), but it keeps the drawings smaller.
base_filters = [
    ">>clean-ladder-filter>>",
    ">>elbow-reduction-filter>>",
]

# The same circuit, with and without the extra toffoli-filter pass.
without = build_circuit(pre_filters=base_filters)
with_toffoli = build_circuit(pre_filters=base_filters + [">>toffoli-filter>>"])

print("| toffoli-filter | T gates | Toffoli gates | Left/right elbows | Measurements |")
for name, qpu in [("Without", without), ("With", with_toffoli)]:
    r = resource_estimator(qpu).resources()
    print(f"| {name:14} | {r['t_gates']: ^7} | {r['toffs']: ^13} | "
          f"{r['gidney_lelbows']:8}/{r['gidney_relbows']:<8} | {r['measurements']: ^12} |")

# And draw them, so the elbow -> Clifford+T compilation is visible, not just tabulated.
print("\nBefore the toffoli-filter (elbows + Toffoli gates):")
without.draw()
print("\nAfter the toffoli-filter (Clifford+T + measurements):")
with_toffoli.draw()

With the pass on, the Toffolis and elbows vanish, compiled into T gates and measurements. The T-count jumps from $7$ to $29$, and the extra $22$ T gates are exactly what our decomposition rule predicts: two Toffoli gates at $7$ T each ($14$) plus two temporary-AND elbow pairs at $4$ T each ($8$). The two measurements come from the right elbow uncomputation.

An alternative scheme involves compiling Toffoli gates into CCZ states. If the architecture of a quantum computer supports distilling CCZ states as the magic states, it allows implementing Toffoli gates directly rather than using T gates. This is why such decompositions depend on the choice of architecture.

Which of these metrics you report is the architecture-dependent call. Collapsing everything to one number, the T-count, is convenient and common in papers. Keeping Toffoli gates separate preserves information that matters for hardware that treats them differently.

> Elbows are the subtler case. A Gidney temporary-AND is asymmetric: the right elbow is uncomputed via a measurement, far more cheaply than the left elbow was computed. On architectures that exploit that asymmetry (PsiQuantum's among them) an elbow's cost profile differs from other gates, so it's worth tracking as a separate resource rather than folding it into the T-count.

So depending on the architecture, you might apply different filters and end up with a different distribution of resources. It's a judgement call, and largely outside the scope of this tutorial, but worth keeping in mind.

## Conclusion

In this kata, we worked with two representations of one algorithm:

- For **algorithm development**, you use the high-level picture to reason about the intent of the operations and how they manipulate state.
- For **resource estimation**, you want to use a picture that represents how your quantum program runs on a fault-tolerant machine, which the high level picture does not achieve. To get an accurate, realistic resource estimate, we apply filters that implement compilation passes in the same way the real device would need and count what comes out.

Filters are how we move between the two representations. 

- Decomposition filters *reveal* the hidden cost of operations, representing more complicated gates as sequences of simpler gates.
- Optimization filters, such as the elbow cancellation, genuinely *reduce* the total cost of running a program.

> Copyright (c) 2026 PsiQuantum